In [1]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# ===============================
# 2. Load train & live datasets
# ===============================
train_df = pd.read_csv("../Dataset/high_salary.csv")
live_df = pd.read_csv("../Dataset/high_salary.live.csv")

# ===============================
# 3. Split X, y (train only)
# ===============================
target_col = "label"                  # change if your target name is different

X = train_df.drop(columns=[target_col, "id"], errors="ignore")
y = train_df[target_col]

# train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ===============================
# 4. Build Preprocessor
# ===============================
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

# Helper to train + eval quickly
def fit_and_eval(model_name, model):
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    print(f"\n=== {model_name} ===")
    print("Accuracy:", round(acc, 4))
    print("F1 Score:", round(f1, 4))
    return pipe, acc, f1

# ===============================
# 5. Models: RF, Logistic, Decision Tree
# ===============================

# Random Forest (tuned / baseline)
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=42
)
rf_pipe, rf_acc, rf_f1 = fit_and_eval("Random Forest", rf_model)

# Logistic Regression
log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs"
)
log_pipe, log_acc, log_f1 = fit_and_eval("Logistic Regression", log_model)

# Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)
dt_pipe, dt_acc, dt_f1 = fit_and_eval("Decision Tree", dt_model)

# ===============================
# 6. Summary table
# ===============================
summary = pd.DataFrame({
    "Model": ["Random Forest", "Logistic Regression", "Decision Tree"],
    "Accuracy": [rf_acc, log_acc, dt_acc],
    "F1 Score": [rf_f1, log_f1, dt_f1]
})
print("\n===== Model Comparison =====")
print(summary)

# ===============================
# 7. (Optional) Predict on live with best model
#    Here I assume Random Forest is best
# ===============================
live_features = live_df.drop(columns=["id"], errors="ignore")
live_pred = rf_pipe.predict(live_features)

submission = pd.DataFrame({
    "id": live_df["id"],
    "prediction": live_pred
})
# submission.to_csv("G19_predictions.live.csv", index=False)


=== Random Forest ===
Accuracy: 0.8246
F1 Score: 0.7976

=== Logistic Regression ===
Accuracy: 0.812
F1 Score: 0.7906

=== Decision Tree ===
Accuracy: 0.7586
F1 Score: 0.7101

===== Model Comparison =====
                 Model  Accuracy  F1 Score
0        Random Forest  0.824641  0.797570
1  Logistic Regression  0.811962  0.790623
2        Decision Tree  0.758612  0.710141
